In [1]:
!pip install -qU llama-index
!pip install -qU llama-index-embeddings-huggingface
!pip install -qU llama-index-retrievers-bm25
!pip install -qU llama-index-postprocessor-flag-embedding-reranker
!pip install -qU sentence-transformers rank_bm25 nest_asyncio

In [ ]:
import os
# --- 1. HUGGING FACE TOKEN (İndirme Hızlandırıcısı) ---
os.environ["HF_TOKEN"] = ""
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import nest_asyncio
nest_asyncio.apply()

import pandas as pd
import asyncio
import gc
import torch
from typing import List

from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter, MarkdownNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.postprocessor.flag_embedding_reranker import FlagEmbeddingReranker

all_results = []

def add_result(group, model_name, mrr, hit_rate):
    all_results.append({
        "Group": group,
        "System": model_name,
        "MRR@5": round(mrr, 4),
        "HitRate@5": round(hit_rate, 4)
    })
    print(f"[{group}] {model_name} -> MRR: {mrr:.4f} | Hit Rate: {hit_rate:.4f}")

# --- GÜNCELLENEN AKILLI VE ESNEK ASENKRON DEĞERLENDİRİCİ ---
async def evaluate_retriever_custom(retriever, queries_dict, relevant_docs_dict, top_k=5, max_concurrent=15):
    sem = asyncio.Semaphore(max_concurrent)
    
    async def process_single_query(query_str, expected_ids):
        if not expected_ids:
            return None
        # Karşılaştırma yaparken uzantı veya boşluk hatalarını önlemek için temizleme yapıyoruz
        expected_ids_clean = [str(eid).strip().split(".")[0] for eid in expected_ids]
        
        async with sem:
            try:
                retrieved_nodes = await retriever.aretrieve(query_str)
                retrieved_nodes = retrieved_nodes[:top_k]
                
                # ID EŞLEŞTİRME GARANTİSİ: Node içerisindeki tüm olası ID kaynaklarını tarıyoruz
                retrieved_doc_ids = []
                for n in retrieved_nodes:
                    possible_ids = []
                    # 1. Olasılık: ref_doc_id (Üst döküman ID'si)
                    if getattr(n.node, "ref_doc_id", None):
                        possible_ids.append(n.node.ref_doc_id)
                    # 2. Olasılık: node_id (Kendi ID'si)
                    if getattr(n.node, "node_id", None):
                        possible_ids.append(n.node.node_id)
                    # 3. Olasılık: Metadata içindeki dosya ismi
                    if n.node.metadata and "file_name" in n.node.metadata:
                        possible_ids.append(n.node.metadata["file_name"])
                    
                    # Bulunan tüm olası ID'leri temizleyip listeye ekle
                    for pid in possible_ids:
                        clean_pid = str(pid).strip().split(".")[0]
                        if clean_pid not in retrieved_doc_ids:
                            retrieved_doc_ids.append(clean_pid)
                
                # Hit Rate Hesapla
                hit = 1.0 if any(exp_id in retrieved_doc_ids for exp_id in expected_ids_clean) else 0.0
                
                # MRR Hesapla
                mrr = 0.0
                for rank, doc_id in enumerate(retrieved_doc_ids):
                    if doc_id in expected_ids_clean:
                        mrr = 1.0 / (rank + 1)
                        break
                return (mrr, hit)
            except Exception as e:
                return None
                
    tasks = [process_single_query(q_str, relevant_docs_dict.get(q_id, [])) for q_id, q_str in queries_dict.items()]
    results = await asyncio.gather(*tasks)
    
    valid_results = [r for r in results if r is not None]
    if not valid_results:
        return 0.0, 0.0
        
    avg_mrr = sum(r[0] for r in valid_results) / len(valid_results)
    avg_hit = sum(r[1] for r in valid_results) / len(valid_results)
    return avg_mrr, avg_hit

In [3]:
import json
import glob
from llama_index.core.evaluation import QueryResponseDataset
from llama_index.core import SimpleDirectoryReader

BASE_PATH = "/kaggle/input/datasets/yekbun/turkish-rag-dataset/rag-dataset"
QA_PATH = os.path.join(BASE_PATH, "benchmark")
DOCS_PATH = os.path.join(BASE_PATH, "stage2_cleaned")

print("1. Dokümanlar (Corpus) yükleniyor...")
reader = SimpleDirectoryReader(input_dir=DOCS_PATH, recursive=True)
documents = reader.load_data()

for doc in documents:
    doc.id_ = doc.metadata["file_name"].split(".")[0] 

print(f"Toplam Doküman Sayısı: {len(documents)}")

print("2. QA (Soru-Cevap) Çiftleri yükleniyor...")
queries = {}
relevant_docs = {}

qa_files = glob.glob(os.path.join(QA_PATH, "*.json"))

for file_path in qa_files:
    try:
        temp_dataset = QueryResponseDataset.from_json(file_path)
        queries.update(temp_dataset.queries)
        for k, v in temp_dataset.relevant_docs.items():
            relevant_docs[k] = relevant_docs.get(k, []) + v
    except Exception:
        with open(file_path, 'r', encoding='utf-8') as f:
            qa_data = json.load(f)
        for idx, item in enumerate(qa_data):
            q_id = item.get("id", item.get("query_id", f"{os.path.basename(file_path)}_{idx}"))
            queries[q_id] = item.get("query", item.get("question", ""))
            exp_docs = item.get("expected_doc_id", item.get("doc_id", []))
            if not isinstance(exp_docs, list):
                exp_docs = [exp_docs]
            relevant_docs[q_id] = exp_docs

# Tüm veri seti devrede
TEST_LIMIT = None 

if TEST_LIMIT and TEST_LIMIT < len(queries):
    print(f"\n⚠️ UYARI: Sadece İLK {TEST_LIMIT} SORU kullanılacak!")
    queries = dict(list(queries.items())[:TEST_LIMIT])
    relevant_docs = {k: relevant_docs.get(k, []) for k in queries.keys()}

print(f"Değerlendirmeye Alınacak Soru Sayısı: {len(queries)}")

1. Dokümanlar (Corpus) yükleniyor...
Toplam Doküman Sayısı: 236
2. QA (Soru-Cevap) Çiftleri yükleniyor...
Değerlendirmeye Alınacak Soru Sayısı: 2132


In [ ]:
print("\n--- GRUP 1 BAŞLIYOR: EMBEDDING KARŞILAŞTIRMASI ---")

embedding_models = {
    "RAG-1": "BAAI/bge-m3",
    "RAG-2": "intfloat/multilingual-e5-large",
    "RAG-3": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
}

Settings.text_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=128)
group1_results = []

for rag_id, model_name in embedding_models.items():
    print(f"\nHesaplanıyor: {rag_id} ({model_name})...")
    gc.collect()
    torch.cuda.empty_cache()

    embed_model = HuggingFaceEmbedding(model_name=model_name)
    Settings.embed_model = embed_model
    
    index = VectorStoreIndex.from_documents(documents)
    retriever = index.as_retriever(similarity_top_k=5)
    
    # Özel ve hızlı evaluator'ı çağır
    mrr, hit_rate = await evaluate_retriever_custom(retriever, queries, relevant_docs, top_k=5, max_concurrent=15)
    
    add_result("Group 1", f"{rag_id} ({model_name.split('/')[-1]})", mrr, hit_rate)
    group1_results.append({"name": model_name, "mrr": mrr})
    
    del index, retriever, embed_model
    Settings.embed_model = None 
    gc.collect()
    torch.cuda.empty_cache()

best_embedding_model = max(group1_results, key=lambda x: x['mrr'])['name']
print(f"\n🏆 GRUP 1 KAZANANI: {best_embedding_model}")

Settings.embed_model = HuggingFaceEmbedding(model_name=best_embedding_model)


--- GRUP 1 BAŞLIYOR: EMBEDDING KARŞILAŞTIRMASI ---

Hesaplanıyor: RAG-1 (BAAI/bge-m3)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[Group 1] RAG-1 (bge-m3) -> MRR: 0.0000 | Hit Rate: 0.0000
Embeddings have been explicitly disabled. Using MockEmbedding.

Hesaplanıyor: RAG-2 (intfloat/multilingual-e5-large)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
print("\n--- GRUP 2 BAŞLIYOR: CHUNK STRATEJİSİ ---")

chunk_strategies = {
    "RAG-4 (256/50)": SentenceSplitter(chunk_size=256, chunk_overlap=50),
    "RAG-5 (1024/200)": SentenceSplitter(chunk_size=1024, chunk_overlap=200),
    "RAG-6 (Header Based)": MarkdownNodeParser()
}

group2_results = []

for strategy_name, splitter in chunk_strategies.items():
    print(f"\nHesaplanıyor: {strategy_name}...")
    Settings.text_splitter = splitter
    
    index = VectorStoreIndex.from_documents(documents)
    retriever = index.as_retriever(similarity_top_k=5)
    
    mrr, hit_rate = await evaluate_retriever_custom(retriever, queries, relevant_docs, top_k=5, max_concurrent=15)
    
    add_result("Group 2", strategy_name, mrr, hit_rate)
    group2_results.append({"name": strategy_name, "splitter": splitter, "mrr": mrr})

best_chunk_strategy = max(group2_results, key=lambda x: x['mrr'])
print(f"\n🏆 GRUP 2 KAZANANI: {best_chunk_strategy['name']}")

Settings.text_splitter = best_chunk_strategy['splitter']
best_index = VectorStoreIndex.from_documents(documents)

In [ ]:
import json
import glob
import os
from llama_index.core.evaluation import QueryResponseDataset
from llama_index.core import SimpleDirectoryReader

# Kaggle ortamındaki dosya yolları 
BASE_PATH = "/kaggle/input/datasets/yekbun/dataset1/dataset"
QA_PATH = os.path.join(BASE_PATH, "benchmark")
DOCS_PATH = os.path.join(BASE_PATH, "stage2_cleaned")

print("1. Dokümanlar (Corpus) yükleniyor...")
reader = SimpleDirectoryReader(input_dir=DOCS_PATH, recursive=True)
documents = reader.load_data()

# DOKÜMAN ID'LERİNİ TEMİZLE (Örn: "m01.md" -> "m01")
for doc in documents:
    doc.id_ = doc.metadata["file_name"].replace(".md", "").replace(".txt", "").strip()

print(f"Toplam Doküman Sayısı: {len(documents)}")

print("2. QA (Soru-Cevap) Çiftleri yükleniyor...")
queries = {}
relevant_docs = {}

qa_files = glob.glob(os.path.join(QA_PATH, "*.json"))

for file_path in qa_files:
    try:
        # Standart LlamaIndex formatı için
        temp_dataset = QueryResponseDataset.from_json(file_path)
        queries.update(temp_dataset.queries)
        for k, v in temp_dataset.relevant_docs.items():
            relevant_docs[k] = relevant_docs.get(k, []) + v
    except Exception:
        # SİZİN JSON FORMATINIZ İÇİN ÖZEL AYARLAR
        with open(file_path, 'r', encoding='utf-8') as f:
            qa_data = json.load(f)
            
        for idx, item in enumerate(qa_data):
            # Benzersiz bir soru ID'si oluştur (dosyaadı_sorunumarası)
            q_id = f"{os.path.basename(file_path)}_{idx}"
            
            # Soruyu çek
            queries[q_id] = item.get("question", item.get("query", ""))
            
            # HATA BURADAYDI! Artık "source_file" anahtarına bakıyoruz.
            exp_docs = item.get("source_file", item.get("expected_doc_id", []))
            
            if not isinstance(exp_docs, list):
                exp_docs = [exp_docs]
                
            # JSON'dan gelen isimleri de temizle (Örn: "m01.md" -> "m01")
            cleaned_exp_docs = [str(doc).replace(".md", "").replace(".txt", "").strip() for doc in exp_docs]
            relevant_docs[q_id] = cleaned_exp_docs

print(f"Toplam Soru Sayısı (Ham Veri): {len(queries)}")

# --- TEST LİMİTİ ---
# Tüm veri seti devrede
TEST_LIMIT = None 

if TEST_LIMIT and TEST_LIMIT < len(queries):
    print(f"\n⚠️ UYARI: Sadece İLK {TEST_LIMIT} SORU kullanılacak!")
    queries = dict(list(queries.items())[:TEST_LIMIT])
    relevant_docs = {k: relevant_docs.get(k, []) for k in queries.keys()}

print(f"Değerlendirmeye Alınacak Soru Sayısı: {len(queries)}")

In [ ]:
from llama_index.core.indices.query.schema import QueryBundle

print("\n--- GRUP 4 BAŞLIYOR: RERANKER ETKİSİ ---")
gc.collect()
torch.cuda.empty_cache()

reranker = FlagEmbeddingReranker(
    top_n=5,
    model="BAAI/bge-reranker-v2-m3"
)

# Reranker'ı Hızlı Evaluator ile uyumlu hale getirmek için özel sınıf
class CustomRerankRetriever:
    def __init__(self, base_retriever, reranker_model):
        self.base_retriever = base_retriever
        self.reranker_model = reranker_model
        
    async def aretrieve(self, query_str):
        self.base_retriever.similarity_top_k = 15 # Önce 15 aday topla
        nodes = await self.base_retriever.aretrieve(query_str)
        # Reranker ile 5'e düşür ve sırala
        return self.reranker_model.postprocess_nodes(nodes, query_bundle=QueryBundle(query_str))

print("\nHesaplanıyor: RAG-9 (Hybrid + Reranker)...")
rerank_retriever = CustomRerankRetriever(hybrid_retriever, reranker)

# DİKKAT: Reranker çok ağır olduğu için max_concurrent=3 yapıyoruz ki GPU patlamasın!
mrr, hit_rate = await evaluate_retriever_custom(rerank_retriever, queries, relevant_docs, top_k=5, max_concurrent=3)
add_result("Group 4", "RAG-9 (Hybrid + Reranker)", mrr, hit_rate)

In [ ]:
df_results = pd.DataFrame(all_results)

print("\n" + "="*60)
print("🏆 TÜM RAG SİSTEMLERİ PERFORMANS TABLOSU (RAG 1 - RAG 9) 🏆")
print("="*60)
display(df_results.sort_values(by=["Group", "MRR@5"], ascending=[True, False]))

df_results.to_csv('rag_evaluation_results.csv', index=False)
print("\nSonuçlar 'rag_evaluation_results.csv' olarak kaydedildi.")